In [0]:
%sql
-- Cargar dimensión: dim_marca
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: automotor_marca_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.dim_marca
SELECT DISTINCT
  automotor_marca_codigo,
  automotor_marca_descripcion
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE automotor_marca_codigo IS NOT NULL
ORDER BY automotor_marca_codigo;

In [0]:
%sql
-- Cargar dimensión: dim_tipo_vehiculo
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: automotor_tipo_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.dim_tipo_vehiculo
SELECT DISTINCT
  automotor_tipo_codigo,
  automotor_tipo_descripcion
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE automotor_tipo_codigo IS NOT NULL
ORDER BY automotor_tipo_codigo;

In [0]:
%sql
-- Cargar dimensión: dim_modelo
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: automotor_modelo_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.dim_modelo
SELECT DISTINCT
  automotor_modelo_codigo,
  automotor_modelo_descripcion
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE automotor_modelo_codigo IS NOT NULL
ORDER BY automotor_modelo_codigo;

In [0]:
%sql
-- Cargar dimensión: dim_geografia
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: registro_seccional_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.dim_geografia
SELECT DISTINCT
  registro_seccional_codigo,
  registro_seccional_provincia
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE registro_seccional_codigo IS NOT NULL
ORDER BY registro_seccional_codigo;

In [0]:
%sql
-- Cargar tabla de hechos: fact_transferencias
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: id_tramite
-- FKs: automotor_marca_codigo, automotor_tipo_codigo, automotor_modelo_codigo, registro_seccional_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.fact_transferencias
SELECT
  -- Primary Key
  id_tramite,
  
  -- Fecha del trámite
  tramite_fecha,
  
  -- Métricas cuantitativas
  automotor_anio_modelo,
  
  -- Foreign Keys a las dimensiones (solo códigos)
  automotor_marca_codigo,
  automotor_tipo_codigo,
  automotor_modelo_codigo,
  registro_seccional_codigo
  
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE id_tramite IS NOT NULL;

## Validaciones de Calidad del Modelo Dimensional

Esta sección verifica la integridad y calidad de los datos en la capa Gold:

* **Conteos de registros**: Verificar que las tablas se crearon correctamente
* **Unicidad de PKs**: Asegurar que las claves primarias sean únicas
* **Integridad referencial**: Verificar que todas las FKs existan en sus dimensiones
* **Valores nulos**: Detectar problemas de calidad en columnas críticas
* **Reconciliación**: Comparar conteos entre Silver y Gold

In [0]:
%sql
-- Validación 1: Conteo de registros en todas las tablas del modelo
-- Permite verificar que las tablas se poblaron correctamente

SELECT 'dim_marca' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.dim_marca
UNION ALL
SELECT 'dim_tipo_vehiculo' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.dim_tipo_vehiculo
UNION ALL
SELECT 'dim_modelo' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.dim_modelo
UNION ALL
SELECT 'dim_geografia' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.dim_geografia
UNION ALL
SELECT 'fact_transferencias' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.fact_transferencias
ORDER BY tabla;

In [0]:
%sql
-- Validación 2: Verificar unicidad de Primary Keys en dimensiones
-- Si hay duplicados, el conteo de PKs distintas será menor al total de registros

SELECT 
  'dim_marca' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT automotor_marca_codigo) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT automotor_marca_codigo) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.dim_marca

UNION ALL

SELECT 
  'dim_tipo_vehiculo' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT automotor_tipo_codigo) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT automotor_tipo_codigo) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.dim_tipo_vehiculo

UNION ALL

SELECT 
  'dim_modelo' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT automotor_modelo_codigo) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT automotor_modelo_codigo) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.dim_modelo

UNION ALL

SELECT 
  'dim_geografia' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT registro_seccional_codigo) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT registro_seccional_codigo) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.dim_geografia

UNION ALL

SELECT 
  'fact_transferencias' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT id_tramite) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT id_tramite) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias;

In [0]:
%sql
-- Validación 3: Detectar valores nulos en Primary Keys y Foreign Keys
-- Estas columnas NO deben contener nulos

SELECT
  'fact_transferencias.id_tramite (PK)' AS columna,
  COUNT(*) - COUNT(id_tramite) AS nulos,
  CASE WHEN COUNT(*) = COUNT(id_tramite) THEN '✓ OK' ELSE '✗ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias

UNION ALL

SELECT
  'fact_transferencias.automotor_marca_codigo (FK)' AS columna,
  COUNT(*) - COUNT(automotor_marca_codigo) AS nulos,
  CASE WHEN COUNT(*) = COUNT(automotor_marca_codigo) THEN '✓ OK' ELSE '⚠ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias

UNION ALL

SELECT
  'fact_transferencias.automotor_tipo_codigo (FK)' AS columna,
  COUNT(*) - COUNT(automotor_tipo_codigo) AS nulos,
  CASE WHEN COUNT(*) = COUNT(automotor_tipo_codigo) THEN '✓ OK' ELSE '⚠ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias

UNION ALL

SELECT
  'fact_transferencias.automotor_modelo_codigo (FK)' AS columna,
  COUNT(*) - COUNT(automotor_modelo_codigo) AS nulos,
  CASE WHEN COUNT(*) = COUNT(automotor_modelo_codigo) THEN '✓ OK' ELSE '⚠ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias

UNION ALL

SELECT
  'fact_transferencias.registro_seccional_codigo (FK)' AS columna,
  COUNT(*) - COUNT(registro_seccional_codigo) AS nulos,
  CASE WHEN COUNT(*) = COUNT(registro_seccional_codigo) THEN '✓ OK' ELSE '⚠ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias;

In [0]:
%sql
-- Validación 4: Verificar integridad referencial entre fact y dimensiones
-- Detecta registros en fact_transferencias que apuntan a códigos inexistentes en las dimensiones

WITH orphan_checks AS (
  SELECT
    'automotor_marca_codigo' AS foreign_key,
    COUNT(DISTINCT f.automotor_marca_codigo) AS codigos_en_fact,
    (
      SELECT COUNT(DISTINCT f2.automotor_marca_codigo)
      FROM workspace.tp_dnrpa_gold.fact_transferencias f2
      LEFT JOIN workspace.tp_dnrpa_gold.dim_marca d ON f2.automotor_marca_codigo = d.automotor_marca_codigo
      WHERE d.automotor_marca_codigo IS NULL AND f2.automotor_marca_codigo IS NOT NULL
    ) AS orphan_records
  FROM workspace.tp_dnrpa_gold.fact_transferencias f
  
  UNION ALL
  
  SELECT
    'automotor_tipo_codigo' AS foreign_key,
    COUNT(DISTINCT f.automotor_tipo_codigo) AS codigos_en_fact,
    (
      SELECT COUNT(DISTINCT f2.automotor_tipo_codigo)
      FROM workspace.tp_dnrpa_gold.fact_transferencias f2
      LEFT JOIN workspace.tp_dnrpa_gold.dim_tipo_vehiculo d ON f2.automotor_tipo_codigo = d.automotor_tipo_codigo
      WHERE d.automotor_tipo_codigo IS NULL AND f2.automotor_tipo_codigo IS NOT NULL
    ) AS orphan_records
  FROM workspace.tp_dnrpa_gold.fact_transferencias f
  
  UNION ALL
  
  SELECT
    'automotor_modelo_codigo' AS foreign_key,
    COUNT(DISTINCT f.automotor_modelo_codigo) AS codigos_en_fact,
    (
      SELECT COUNT(DISTINCT f2.automotor_modelo_codigo)
      FROM workspace.tp_dnrpa_gold.fact_transferencias f2
      LEFT JOIN workspace.tp_dnrpa_gold.dim_modelo d ON f2.automotor_modelo_codigo = d.automotor_modelo_codigo
      WHERE d.automotor_modelo_codigo IS NULL AND f2.automotor_modelo_codigo IS NOT NULL
    ) AS orphan_records
  FROM workspace.tp_dnrpa_gold.fact_transferencias f
  
  UNION ALL
  
  SELECT
    'registro_seccional_codigo' AS foreign_key,
    COUNT(DISTINCT f.registro_seccional_codigo) AS codigos_en_fact,
    (
      SELECT COUNT(DISTINCT f2.registro_seccional_codigo)
      FROM workspace.tp_dnrpa_gold.fact_transferencias f2
      LEFT JOIN workspace.tp_dnrpa_gold.dim_geografia d ON f2.registro_seccional_codigo = d.registro_seccional_codigo
      WHERE d.registro_seccional_codigo IS NULL AND f2.registro_seccional_codigo IS NOT NULL
    ) AS orphan_records
  FROM workspace.tp_dnrpa_gold.fact_transferencias f
)
SELECT
  foreign_key,
  codigos_en_fact,
  orphan_records,
  CASE WHEN orphan_records = 0 THEN '✓ OK' ELSE '✗ HAY REGISTROS HUÉRFANOS' END AS resultado
FROM orphan_checks;

In [0]:
%sql
-- Validación 5: Reconciliación de conteos entre Silver y Gold
-- Verifica que no se hayan perdido registros en la transformación

WITH conteos AS (
  SELECT
    (SELECT COUNT(*) FROM workspace.tp_dnrpa_silver.silver_transferencias WHERE id_tramite IS NOT NULL) AS registros_silver,
    (SELECT COUNT(*) FROM workspace.tp_dnrpa_gold.fact_transferencias) AS registros_gold
)
SELECT
  registros_silver,
  registros_gold,
  registros_silver - registros_gold AS diferencia,
  ROUND(TRY_DIVIDE(registros_gold * 100.0, registros_silver), 2) AS porcentaje_carga,
  CASE 
    WHEN registros_silver = 0 THEN '✗ SIN DATOS EN SILVER'
    WHEN registros_silver = registros_gold THEN '✓ COINCIDENCIA EXACTA'
    WHEN registros_gold >= (registros_silver * 0.99) THEN '⚠ COINCIDENCIA ACEPTABLE (>99%)'
    ELSE '✗ DISCREPANCIA SIGNIFICATIVA'
  END AS resultado
FROM conteos;